<a href="https://colab.research.google.com/github/sk27110/basic_ml_hse/blob/task_4/scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Парсинг Лента.ру

In [ ]:
import requests as rq
from bs4 import BeautifulSoup as bs
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from IPython import display
import json
import time

In [ ]:
blocs = {'russia': 1, 'economy': 4, 'security_forces': 37, 'ussr': 3, 'sport': 8, 'health': 87, 'tourism': 48, 'science': 5 } #спорта нет, его отдельно доделать
periods = [{'from': '2023-01-01', 'to': '2024-01-01'}, {'from': '2024-01-02', 'to': '2025-01-01'}]

In [ ]:
!pip install httpx

Собираем данные кроме строительства

In [ ]:
import httpx

data = dict()


for key, value in blocs.items():
  item = []
  for period in periods:
    time.sleep(1)
    From = period['from']
    To = period['to']
    url = f"https://lenta.ru/search/v2/process?from=0&size=800&sort=2&title_only=0&domain=1&modified%2Cformat=yyyy-MM-dd&bloc={value}&type=1&modified%2Cfrom={From}&modified%2Cto={To}"
    response = httpx.get(url)
    while response.status_code != 200:
      response = httpx.get(url)
    item+=response.json()['matches']
  data[key] = item



Тут собираем про строительство уже с помощью запроса в query

In [ ]:
item=[]
for period in periods:
    time.sleep(1)
    From = period['from']
    To = period['to']
    url = f"https://lenta.ru/search/v2/process?query=строительство&from=0&size=800&sort=2&title_only=0&domain=1&modified%2Cformat=yyyy-MM-dd&type=1&modified%2Cfrom={From}&modified%2Cto={To}"
    response = httpx.get(url)
    while response.status_code != 200:
      response = httpx.get(url)
    item += response.json()['matches']

data['construction'] = item

In [ ]:
rubrics = {'russia': 0, 'economy': 1, 'security_forces': 2, 'ussr': 3, 'sport': 4, 'health': 5, 'construction': 6, 'tourism': 7, 'science': 8 } #спорта нет, его отдельно доделать

In [ ]:
df = []

for key, value in data.items():
  for item in value:
    item['rubrics'] = rubrics[key]
    df.append(item)

In [ ]:
df = pd.DataFrame(df)

In [ ]:
df.head()

,docid,url,title,modified,lastmodtime,type,domain,status,part,bloc,tags,image_url,pubdate,text,rightcol,snippet,rubrics
0,1551172,https://lenta.ru/news/2024/01/01/massovye-zade...,Массовые задержания мигрантов произошли в ново...,1704142249,1704142249,1,1,0,1,1,[2],https://icdn.lenta.ru/images/2024/01/01/23/202...,1704142249,Фото: Nikolay Gyngazov / Globallookpress.com М...,Массовые задержания мигрантов произошли в ново...,Фото: Nikolay Gyngazov / Globallookpress....«К...,0
1,1551168,https://lenta.ru/news/2024/01/01/v-gosdume-ots...,В Госдуме оценили новогоднее обращение посольс...,1704140440,1704140440,1,1,0,1,1,[1],https://icdn.lenta.ru/images/2024/01/01/23/202...,1704140440,Василий Пискарев Фото: Russian State Duma Phot...,В Госдуме оценили новогоднее обращение посольс...,"Василий Пискарев Фото: Russian State ..., кто ...",0
2,1551166,https://lenta.ru/news/2024/01/01/gorod_/,В Чечне появился восьмой город,1704139560,1704139819,1,1,0,0,1,[2],https://icdn.lenta.ru/images/2024/01/01/23/202...,1704139560,Рамзан Кадыров Фото: Ministry of Culture / Glo...,В Чечне появился восьмой город,Рамзан Кадыров Фото: Ministry of Culture ... К...,0
3,1551160,https://lenta.ru/news/2024/01/01/sick/,Обращение Зеленского к Крыму назвали больными ...,1704137280,1704137729,1,1,0,0,1,[1],https://icdn.lenta.ru/images/2024/01/01/22/202...,1704137280,Владимир Зеленский Фото: Ukrainian Presidentia...,Обращение Зеленского к Крыму назвали больными ...,Владимир Зеленский Фото: Ukrainian ... новых р...,0
4,1551162,https://lenta.ru/news/2024/01/01/v-rossiyskom-...,В российском регионе предупредили о распростра...,1704137016,1704137016,1,1,0,0,1,[2],,1704137016,Фото: Globallookpress.com Марина Совина Минист...,В российском регионе предупредили о распростра...,Фото: Globallookpress.com Марина Совина ... в ...,0


In [ ]:
df.to_csv('dz4_data.csv', index=False, encoding='utf-8-sig') #сохранимся на всякий случай

Удалим все ненужные фичи, нас интересует только текст

In [ ]:
cleaned_df = df.drop(['docid', 'modified'	,'lastmodtime',	'type',	'domain','status'	,'part'	,'bloc', 'url', 'tags', 'image_url', 'pubdate', 'snippet', 'rightcol'], axis = 1)

In [ ]:
cleaned_df['text'] = cleaned_df.apply(lambda row: row['title'] + ' ' + row['text'], axis=1)

In [ ]:
cleaned_df.drop('title', inplace=True, axis=1)

In [ ]:
cleaned_df.to_csv('dz4_cleaned_data.csv', index=False, encoding='utf-8-sig') #сохранимся на всякий случай